# Assignment 11: Fine-Tuning GPT-2 for a Specific Writing Style

**Name:** Dakshraj Singh Chandawat  
**Registration No.:** 2411021240050  
**Programme:** BCA (AI/ML), Semester V — Alliance University  
**Subject:** Generative AI

## 1. Objective

The purpose of this assignment is to study how fine-tuning allows a pretrained language model to adjust its generated text toward a particular writing style. A small GPT-2 model is fine-tuned using Shakespearean dramatic dialogue, and its generated text is then compared with the output of the original GPT-2 model using the same prompt from Assignment 2.

## 2. Approach

| Step | Description |
|---|---|
| Dataset collection | Approximately 250 lines of Shakespearean dialogue selected from the public-domain Tiny Shakespeare dataset |
| Dataset cleaning | Speaker labels, empty lines, unnecessary spaces, and non-linguistic content are filtered out |
| Fine-tuning | The 124M-parameter small GPT-2 model is trained using Hugging Face `Trainer` |
| Generation | An identical prompt is supplied to both the original and fine-tuned models |
| Comparison | The outputs are compared using side-by-side examples, style measurements, and written observations |

### 2.1 Why Shakespearean Dialogue?

Shakespeare's language has a distinctive and easily recognisable pattern. It commonly includes older pronouns such as *thou*, *thee*, and *thy*, short poetic lines, unusual word arrangements, and a dramatic tone. Because these characteristics are quite noticeable, they provide a clear way to observe the effect of fine-tuning on the model's writing style.

## 3. Environment Setup

### 3.1 Installing Required Libraries

In [ ]:
!pip install -q transformers datasets accelerate torch pandas

### 3.2 Importing the Required Libraries

In [ ]:
import os
import re
import random
import urllib.request

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

### 3.3 Choosing the Device and Setting the Random Seed

Using a fixed random seed helps make both the training process and generated results more consistent and reproducible.

In [ ]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device in use:", device)

## 4. Dataset Collection

### 4.1 Data Source

The training text comes from the **Tiny Shakespeare** corpus, which is a public-domain collection containing dialogue from Shakespeare's plays. Only the beginning portion of *Coriolanus* is selected so that the resulting dataset remains within the required range of 200–300 lines.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(DATA_URL, "tinyshakespeare_full.txt")

with open("tinyshakespeare_full.txt", "r", encoding="utf-8") as f:
    full_text = f.read()

print("Total characters in full corpus:", len(full_text))
print("Total lines in full corpus:", len(full_text.splitlines()))

### 4.2 Inspecting the Raw Text

In the original text, speaker names appear on separate lines, much like usernames in a conversation, and blank lines occur between speeches. These elements are not useful for learning the intended writing style, so they need to be removed during preprocessing.

In [ ]:
raw_lines = full_text.splitlines()
print("\n".join(raw_lines[:25]))

## 5. Dataset Cleaning

### 5.1 Cleaning Rules

| Unwanted content | Example | Action |
|---|---|---|
| Speaker labels (usernames) | `First Citizen:` | Deleted |
| Empty lines | blank lines separating speeches | Deleted |
| Unnecessary spaces and tabs | repeated spaces | Reduced to a single space |
| Non-linguistic content | timestamps, URLs, bracketed labels, symbols | Removed when encountered |
| Extremely short fragments | lines containing fewer than 2 characters | Discarded |

### 5.2 Cleaning Function

In [ ]:
SPEAKER_PATTERN = re.compile(r"^[A-Z][A-Za-z ]*:$")
TIMESTAMP_PATTERN = re.compile(r"\[?\d{1,2}:\d{2}(:\d{2})?\s*(AM|PM|am|pm)?\]?")
URL_PATTERN = re.compile(r"https?://\S+")
TAG_PATTERN = re.compile(r"<[^>]+>|\[[^\]]*\]")
SYMBOL_PATTERN = re.compile(r"[^A-Za-z0-9\s.,;:!?'\-]")


def clean_line(line):
    text = line.strip()
    if not text or SPEAKER_PATTERN.match(text):
        return None
    text = TIMESTAMP_PATTERN.sub("", text)
    text = URL_PATTERN.sub("", text)
    text = TAG_PATTERN.sub("", text)
    text = SYMBOL_PATTERN.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) < 2:
        return None
    return text

### 5.3 Creating the Final Dataset

Each source line is processed individually, and the cleaned lines are collected until the target of exactly 250 usable lines is reached.

In [ ]:
TARGET_LINES = 250

clean_lines = []
raw_used = 0
for line in raw_lines:
    raw_used += 1
    cleaned = clean_line(line)
    if cleaned:
        clean_lines.append(cleaned)
    if len(clean_lines) == TARGET_LINES:
        break

print("Raw lines read:", raw_used)
print("Clean lines kept:", len(clean_lines))
print("Lines removed during cleaning:", raw_used - len(clean_lines))

### 5.4 Storing the Raw and Cleaned Datasets

The raw and processed versions are both stored so that they can be included with the notebook during submission.

In [ ]:
with open("original_dataset_raw.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(raw_lines[:raw_used]))

with open("cleaned_dataset.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(clean_lines))

print("Saved: original_dataset_raw.txt")
print("Saved: cleaned_dataset.txt")

### 5.5 Comparison Before and After Cleaning

In [ ]:
print("BEFORE CLEANING")
print("-" * 60)
print("\n".join(raw_lines[:15]))
print()
print("AFTER CLEANING")
print("-" * 60)
print("\n".join(clean_lines[:10]))

### 5.6 Dataset Statistics

In [ ]:
words = " ".join(clean_lines).split()
stats = pd.DataFrame({
    "Metric": ["Number of lines", "Number of words", "Unique words", "Average words per line"],
    "Value": [len(clean_lines), len(words), len(set(w.lower() for w in words)), round(len(words) / len(clean_lines), 2)],
})
stats

## 6. Loading the Original GPT-2 Model

### 6.1 Tokenizer and Model

Since GPT-2 does not define a separate padding token, its end-of-sequence token is assigned for use as the padding token.

In [ ]:
MODEL_NAME = "gpt2"

tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
base_model.eval()

print("Model:", MODEL_NAME)
print("Parameters:", f"{sum(p.numel() for p in base_model.parameters()):,}")

### 6.2 Text Generation Function

Both models use identical generation parameters. This ensures that the comparison mainly reflects the effect of fine-tuning rather than differences in the generation setup.

In [ ]:
PROMPT = "The future of artificial intelligence will"

GEN_SETTINGS = dict(
    max_new_tokens=80,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)


def generate_text(model, prompt, seed=SEED):
    set_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            **GEN_SETTINGS,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

### 6.3 Output from the Base Model

The prompt **"The future of artificial intelligence will"** is reused from Assignment 2 so that the original GPT-2 output can be compared consistently with the fine-tuned model.

In [ ]:
base_outputs = [generate_text(base_model, PROMPT, seed=s) for s in [42, 7, 101]]

for i, text in enumerate(base_outputs, start=1):
    print(f"BASE MODEL — SAMPLE {i}")
    print("-" * 60)
    print(text)
    print()

## 7. Preparing the Dataset for Training

### 7.1 Tokenising the Text

The cleaned dialogue lines are combined into a continuous text sequence and then transformed into tokens that can be processed by GPT-2.

In [ ]:
training_text = "\n".join(clean_lines)
raw_dataset = Dataset.from_dict({"text": [training_text]})


def tokenize(batch):
    return tokenizer(batch["text"])


tokenized_dataset = raw_dataset.map(tokenize, batched=True, remove_columns=["text"])
print("Total tokens:", len(tokenized_dataset[0]["input_ids"]))

### 7.2 Dividing the Tokens into Fixed-Length Blocks

The complete token sequence is separated into groups containing 64 tokens. Every group acts as a training example. The labels match the input tokens because GPT-2 is trained as a next-token prediction model.

In [ ]:
BLOCK_SIZE = 64


def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated["input_ids"]) // BLOCK_SIZE) * BLOCK_SIZE
    result = {
        k: [t[i : i + BLOCK_SIZE] for i in range(0, total_length, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result


lm_dataset = tokenized_dataset.map(group_texts, batched=True)
print("Training blocks:", len(lm_dataset))

### 7.3 Data Collator

Setting `mlm=False` indicates that GPT-2 should be treated as a causal, left-to-right language model rather than a masked language model.

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 8. Fine-Tuning with the Hugging Face Trainer

### 8.1 Training Configuration

| Hyperparameter | Value | Reason |
|---|---|---|
| Epochs | 10 | Multiple passes help the small dataset influence the learned style |
| Batch size | 4 | Keeps memory usage manageable |
| Learning rate | 5e-5 | A commonly used fine-tuning rate for GPT-2 |
| Weight decay | 0.01 | Helps limit overfitting |
| Warmup steps | 10 | Provides a more gradual beginning to training |

In [ ]:
OUTPUT_DIR = "gpt2-shakespeare"

ft_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=10,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=ft_model,
    args=training_args,
    train_dataset=lm_dataset,
    data_collator=data_collator,
)

### 8.2 Training the Fine-Tuned Model

In [ ]:
train_result = trainer.train()
print("Final training loss:", round(train_result.training_loss, 4))

### 8.3 Training Loss Curve

A downward trend in the loss indicates that the model is progressively learning the patterns present in the Shakespearean training text.

In [ ]:
import matplotlib.pyplot as plt

loss_log = [(log["step"], log["loss"]) for log in trainer.state.log_history if "loss" in log]
steps, losses = zip(*loss_log)

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, marker="o")
plt.title("Training Loss During Fine-Tuning")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 8.4 Saving the Fine-Tuned Model Checkpoint

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Fine-tuned model saved to:", OUTPUT_DIR)
print("Files:", os.listdir(OUTPUT_DIR))

## 9. Generating Text with the Fine-Tuned Model

The prompt, random seeds, and generation parameters are kept the same as those used in Section 6.3 to make the comparison fair.

In [ ]:
ft_model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR).to(device)
ft_model.eval()

ft_outputs = [generate_text(ft_model, PROMPT, seed=s) for s in [42, 7, 101]]

for i, text in enumerate(ft_outputs, start=1):
    print(f"FINE-TUNED MODEL — SAMPLE {i}")
    print("-" * 60)
    print(text)
    print()

## 10. Comparing the Results

### 10.1 Side-by-Side Model Outputs

In [ ]:
for i, (base, ft) in enumerate(zip(base_outputs, ft_outputs), start=1):
    print("=" * 70)
    print(f"SAMPLE {i}   |   PROMPT: {PROMPT}")
    print("=" * 70)
    print("ORIGINAL GPT-2:")
    print(base)
    print()
    print("FINE-TUNED GPT-2:")
    print(ft)
    print()

### 10.2 Quantifying Style Differences

To make the comparison more objective, several simple characteristics are calculated for the generated text from both models:

- **Archaic words per 100 words:** counts terms such as *thou, thee, thy, hath, doth, 'tis, nay,* and *ay*
- **Average words per line:** helps identify the short, verse-like structure associated with Shakespearean dialogue
- **Number of line breaks:** indicates whether the text resembles dialogue formatting or a continuous paragraph
- **Exclamations and questions:** helps measure the dramatic nature of the generated speech
- **Modern technical words:** counts terms such as *technology, data, computer,* and *system*

In [ ]:
ARCHAIC = {"thou", "thee", "thy", "thine", "hath", "doth", "art", "ye", "nay", "ay",
           "'tis", "tis", "o", "wherefore", "hence", "shalt", "wilt", "methinks", "lord", "good"}
MODERN = {"technology", "data", "computer", "computers", "system", "systems", "software",
          "machine", "machines", "research", "human", "humans", "world", "future", "intelligence"}


def style_metrics(text):
    generated = text[len(PROMPT):]
    tokens = re.findall(r"[A-Za-z']+", generated.lower())
    n = max(len(tokens), 1)
    lines = [l for l in generated.split("\n") if l.strip()]
    return {
        "Archaic words per 100 words": round(100 * sum(t in ARCHAIC for t in tokens) / n, 2),
        "Modern/technical words per 100 words": round(100 * sum(t in MODERN for t in tokens) / n, 2),
        "Average words per line": round(n / max(len(lines), 1), 2),
        "Line breaks": generated.count("\n"),
        "Exclamations and questions": generated.count("!") + generated.count("?"),
    }


base_df = pd.DataFrame([style_metrics(t) for t in base_outputs]).mean().round(2)
ft_df = pd.DataFrame([style_metrics(t) for t in ft_outputs]).mean().round(2)
data_df = pd.Series(style_metrics(PROMPT + "\n".join(clean_lines[:60])))

comparison = pd.DataFrame({
    "Original GPT-2": base_df,
    "Fine-Tuned GPT-2": ft_df,
    "Training Data (reference)": data_df,
})
comparison

### 10.3 Visualising the Comparison

In [ ]:
comparison[["Original GPT-2", "Fine-Tuned GPT-2"]].plot(kind="barh", figsize=(9, 4.5))
plt.title("Style Features: Original vs Fine-Tuned GPT-2 (average of 3 samples)")
plt.xlabel("Value")
plt.tight_layout()
plt.show()

### 10.4 Saving the Generated Outputs

In [ ]:
with open("model_outputs_comparison.txt", "w", encoding="utf-8") as f:
    f.write(f"PROMPT: {PROMPT}\n")
    f.write(f"GENERATION SETTINGS: {GEN_SETTINGS}\n\n")
    for i, (base, ft) in enumerate(zip(base_outputs, ft_outputs), start=1):
        f.write(f"===== SAMPLE {i} =====\n")
        f.write("ORIGINAL GPT-2:\n" + base + "\n\n")
        f.write("FINE-TUNED GPT-2:\n" + ft + "\n\n")
    f.write("===== STYLE METRICS =====\n")
    f.write(comparison.to_string())

comparison.to_csv("style_comparison.csv")
print("Saved: model_outputs_comparison.txt")
print("Saved: style_comparison.csv")

## 11. Analysis

### 11.1 Difference 1 — Vocabulary and Language Era

The **original GPT-2** generally continues the prompt using contemporary English. Since the prompt concerns artificial intelligence, it tends to discuss the topic using terms such as *technology*, *data*, *systems*, and *humans*, producing something closer to a modern article or blog.

The **fine-tuned GPT-2** moves toward an Early Modern English style. Words such as *thou*, *thee*, *thy*, *hath*, *'tis*, and *nay* become more common even though the prompt refers to a modern subject. This change is reflected by the higher value for "archaic words per 100 words" and the lower value for "modern/technical words" shown in the comparison table.

### 11.2 Difference 2 — Structure and Line Length

The **original GPT-2** tends to generate longer prose sentences that appear as continuous text, often with very few line breaks.

The **fine-tuned GPT-2** produces shorter, verse-like lines and uses more line breaks, which resembles the dialogue format present in the training material. As a result, it shows more line breaks and a smaller average number of words per line.

### 11.3 Difference 3 — Tone and Voice

The **original GPT-2** usually maintains a neutral and informative voice, discussing artificial intelligence from an outside perspective.

The **fine-tuned GPT-2** adopts a more theatrical and conversational style. It may directly address another person, include rhetorical questions and exclamations, and sound more like a character delivering a stage speech. It can also move away from the original AI topic because the model gives greater importance to the stylistic patterns learned during fine-tuning.

### 11.4 Summary Table

| Aspect | Original GPT-2 | Fine-Tuned GPT-2 |
|---|---|---|
| Vocabulary | Contemporary and technical | Archaic and Shakespearean |
| Sentence structure | Longer prose-style paragraphs | Short, verse-like lines |
| Tone | Neutral and explanatory | Dramatic and theatrical |
| Point of view | Descriptive, often third-person | Directly addresses a listener |
| Relevance to prompt | More focused on the AI subject | More likely to shift toward dramatic dialogue |

### 11.5 Observations on Training

- The training loss gradually decreased, indicating that the model was adapting to the patterns and style of the Shakespearean dataset.
- Since the dataset contains only 250 lines, the model is more likely to **overfit**, which can result in repeated phrases or character-like expressions that closely resemble the training examples.
- Using more training data, reducing the number of epochs, or introducing a validation split could help reduce memorisation while still preserving the desired writing style.

## 12. Conclusion

Fine-tuning the small GPT-2 model with only 250 lines of Shakespearean dialogue produced a clear change in its generated writing style. When given the same prompt and generation settings, the original GPT-2 generated modern and technical prose, while the fine-tuned version produced shorter, more archaic, and more dramatic text resembling the style of its training data.

This experiment demonstrates that fine-tuning does not require teaching the model an entirely new language. Instead, it modifies how the existing GPT-2 knowledge is expressed so that the generated text better follows the vocabulary, structure, and tone represented in a particular dataset. Even a relatively small but properly cleaned dataset can therefore create a noticeable stylistic shift.

## 13. Submission Files

| Requirement | File |
|---|---|
| Training notebook | `Assignment_11_GPT2_Fine_Tuning.ipynb` |
| Fine-tuned model checkpoint | `gpt2-shakespeare/` |
| Original dataset | `original_dataset_raw.txt` |
| Cleaned dataset | `cleaned_dataset.txt` |
| Base-model and fine-tuned output | `model_outputs_comparison.txt` |
| Before-and-after comparison | Section 10 of this notebook and `style_comparison.csv` |